In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd


# ## Step 0: Load Data & Encoding
# One-hot encoding and NA fill. No outlier removal here.
data = pd.read_csv("analysis_data.csv")
data = data.drop_duplicates()
data_encoded = pd.get_dummies(data, drop_first=True)
data_encoded = data_encoded.fillna(0)

X = data_encoded.drop(columns=['monthly_spend'])
y = data_encoded['monthly_spend']


# ## Step 1: Train–Test Split (70/30)
# A larger test size was used during experimentation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=617
)


# ## Step 2: XGBoost Model Setup
# Manually tuned parameters. Still overfit due to small dataset
# and absence of scaling.
model = XGBRegressor(
    n_estimators=900,
    learning_rate=0.02,
    max_depth=5,
    min_child_weight=6,
    subsample=0.85,
    colsample_bytree=0.75,
    reg_lambda=1.5,
    reg_alpha=0.4,
    gamma=0.3,
    tree_method="hist",
    random_state=617
)


# ## Step 3: Train & Evaluate
model.fit(X_train, y_train)
pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE:", rmse)

# ## Why This Model Failed
# - Overfitting on small dataset
# - Sensitive to unscaled numeric features
# - Outliers were not removed
# - ElasticNet captured structure with less variance